YOLOv8m + ECA+CBAM

Environment

In [1]:
# Detects whether the notebook is running on Kaggle, Colab, or local Jupyter and sets ROOT and OUTPUT_DIR accordingly.
import os, sys

ON_KAGGLE = os.path.exists("/kaggle/input")
ON_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
if ON_KAGGLE:
    ROOT = "/kaggle/working"
elif ON_COLAB:
    ROOT = "/content"
else:
    ROOT = "."
print("Running on:", "KAGGLE" if ON_KAGGLE else "COLAB" if ON_COLAB else "LOCAL")
OUTPUT_DIR = os.path.join(ROOT, "attention_results")
SAVE_DIR = os.path.join(ROOT, "saved_models")
for d in [OUTPUT_DIR, SAVE_DIR]:
    os.makedirs(d, exist_ok=True)
print(f"Output : {OUTPUT_DIR}")
print(f"Models : {SAVE_DIR}")


Running on: LOCAL
Output : ./attention_results
Models : ./saved_models


In [2]:
# Installs kagglehub, matplotlib, pillow and numpy.
!pip install kagglehub matplotlib pillow numpy -q

Verify Environment

In [3]:
# Prints package versions and confirms GPU availability, device name and VRAM.
import torch, numpy as np, pandas as pd, cv2, random, gc

print(f"PyTorch : {torch.__version__}")
print(f"OpenCV  : {cv2.__version__}")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device  : {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
try:
    import ultralytics, seaborn, tqdm, kagglehub, yaml

    print("All packages OK")
except ImportError as e:
    print(f"Missing: {e}")
    print("Run: pip install ultralytics seaborn tqdm kagglehub pyyaml")


PyTorch : 2.10.0+cu128
OpenCV  : 4.13.0
Device  : cuda
GPU     : NVIDIA GeForce RTX 3090
VRAM    : 25.4 GB
All packages OK


Config

In [4]:
# Defines the training and evaluation config: two-phase epochs and learning rates, batch size, image size, weight decay, worker count, seeds, CBAM hyperparameters, and the confidence sweep grid.
EPOCHS_FROZEN = 10
EPOCHS_FULL = 40
BATCH = 16
IMG_SIZE = 640
LR_FROZEN = 1e-3
LR_FULL = 2e-4
WEIGHT_DECAY = 5e-4


NUM_WORKERS = 2

# Multi-seed
SEEDS = [42, 123, 456]

# Attention
CBAM_REDUCTION = 16
CBAM_KERNEL = 7

# Evaluation
CONF_SWEEP = [
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
    0.35,
    0.40,
    0.45,
    0.50,
    0.55,
    0.60,
    0.65,
    0.70,
]
CONF_DEFAULT = 0.25
IOU_THRESH = 0.5

 Dataset + MD5 Deduplication

In [5]:
# Downloads the three Kaggle datasets, loads every annotation through the unified loader, and removes duplicates by MD5 hash before any split.
import kagglehub, shutil, yaml
from pathlib import Path
import xml.etree.ElementTree as ET

print("Downloading datasets ...")
path_1 = kagglehub.dataset_download("chitholian/annotated-potholes-dataset")
path_2 = kagglehub.dataset_download("andrewmvd/pothole-detection")
path_3 = kagglehub.dataset_download("ashishkumarak/training-setzip")
DATASET_ROOTS = {"chitholian": path_1, "andrewmvd": path_2, "ashishkumar": path_3}
print("Datasets ready")


def load_annotated_potholes(root):
    root = Path(root)
    records = []
    for img_path in list(root.rglob("*.jpg")) + list(root.rglob("*.png")):
        xml_path = img_path.with_suffix(".xml")
        if not xml_path.exists():
            xml_path = img_path.parent.parent / "annotations" / (img_path.stem + ".xml")
        gt_boxes = []
        if xml_path.exists():
            try:
                tree = ET.parse(xml_path)
                for obj in tree.findall("object"):
                    bb = obj.find("bndbox")
                    gt_boxes.append(
                        [
                            float(bb.find("xmin").text),
                            float(bb.find("ymin").text),
                            float(bb.find("xmax").text),
                            float(bb.find("ymax").text),
                        ]
                    )
            except Exception:
                pass
        records.append({"image_path": img_path, "gt_boxes": gt_boxes})
    return records


import pandas as pd


def load_ashishkumar_csv(root):
    root = Path(root)
    df = pd.read_csv(root / "train" / "labels.csv")
    grouped = df.groupby("ImageID")
    records = []
    for img_path in sorted((root / "train" / "images").glob("*.jpg")):
        gt_boxes = []
        if img_path.name in grouped.groups:
            for _, row in grouped.get_group(img_path.name).iterrows():
                gt_boxes.append(
                    [
                        float(row["XMin"]),
                        float(row["YMin"]),
                        float(row["XMax"]),
                        float(row["YMax"]),
                    ]
                )
        records.append({"image_path": img_path, "gt_boxes": gt_boxes})
    return records


all_records = []
for name, root in DATASET_ROOTS.items():
    recs = (
        load_ashishkumar_csv(root)
        if name == "ashishkumar"
        else load_annotated_potholes(root)
    )
    print(
        f"  {name}: {len(recs)} images ({sum(len(r['gt_boxes']) for r in recs)} gt boxes)"
    )
    all_records.extend(recs)

print(f"Before dedup: {len(all_records)}")


import numpy as np
from PIL import Image

NORM_SIZE = (64, 64)
DEDUP_THRESHOLD = 1.0


def normalized_pixels(path):
    with Image.open(path) as img:
        return np.asarray(
            img.convert("L").resize(NORM_SIZE, Image.LANCZOS), dtype=np.float32
        ).ravel()


annotated_only = [r for r in all_records if r["gt_boxes"]]
print("Computing normalized pixel arrays for dedup ...")
all_arrs = np.stack([normalized_pixels(r["image_path"]) for r in annotated_only])

keep_mask = np.ones(len(annotated_only), dtype=bool)
seen_arrs = []
for i in range(len(annotated_only)):
    if not keep_mask[i]:
        continue
    if seen_arrs:
        diffs = np.abs(np.stack(seen_arrs) - all_arrs[i]).mean(axis=1)
        if diffs.min() < DEDUP_THRESHOLD:
            keep_mask[i] = False
            continue
    seen_arrs.append(all_arrs[i])

records = [r for r, keep in zip(annotated_only, keep_mask) if keep]
print(f"After dedup: {len(records)}")
print(f"Duplicates removed: {len(all_records) - len(records)}")


Datasets ready
  chitholian: 665 images (1740 gt boxes)
  andrewmvd: 665 images (1740 gt boxes)
  ashishkumar: 674 images (1371 gt boxes)
Before dedup: 2004
Computing normalized pixel arrays for dedup ...
After dedup: 926
Duplicates removed: 1078


Build YOLO Dataset (fixed split, seed=42)

In [6]:
# Shuffles the deduplicated records under seed 42, splits 80/20, and writes the YOLO-format image and label directories plus data.yaml.
random.seed(42)
np.random.seed(42)
random.shuffle(records)
split_idx = int(len(records) * 0.8)
train_recs, val_recs = records[:split_idx], records[split_idx:]
print(f"Train: {len(train_recs)}  Val: {len(val_recs)}")

YOLO_DIR = os.path.join(ROOT, "yolo_dataset")
DATA_YAML = f"{YOLO_DIR}/data.yaml"
for split in ["images/train", "images/val", "labels/train", "labels/val"]:
    os.makedirs(f"{YOLO_DIR}/{split}", exist_ok=True)


def convert_to_yolo(rec_list, split):
    written = 0
    for rec in rec_list:
        img = cv2.imread(str(rec["image_path"]))
        if img is None:
            continue
        h, w = img.shape[:2]
        dst = f"{YOLO_DIR}/images/{split}/{rec['image_path'].name}"
        if not os.path.exists(dst):
            shutil.copy(str(rec["image_path"]), dst)
        with open(f"{YOLO_DIR}/labels/{split}/{rec['image_path'].stem}.txt", "w") as f:
            for box in rec["gt_boxes"]:
                x1, y1, x2, y2 = box
                f.write(
                    f"0 {((x1 + x2) / 2) / w:.6f} {((y1 + y2) / 2) / h:.6f} "
                    f"{(x2 - x1) / w:.6f} {(y2 - y1) / h:.6f}\n"
                )
        written += 1
    return written


n_tr = convert_to_yolo(train_recs, "train")
n_va = convert_to_yolo(val_recs, "val")
with open(DATA_YAML, "w") as f:
    yaml.dump(
        {
            "path": YOLO_DIR,
            "train": "images/train",
            "val": "images/val",
            "nc": 1,
            "names": ["pothole"],
        },
        f,
    )
print(f"YOLO dataset: train={n_tr}  val={n_va}")


Train: 740  Val: 186
YOLO dataset: train=740  val=186


 Attention Modules

In [7]:
# Defines ECA, the CBAM channel and spatial sub-modules, and the three attention variants ECA_Only, CBAM_Only and ECA_CBAM, each wrapped in a zero-initialized learnable residual gate.
import torch, torch.nn as nn, math


class ECA(nn.Module):
    def __init__(self, in_channels, gamma=2, b=1):
        super().__init__()
        t = int(abs(math.log2(in_channels) / gamma) + b / gamma)
        k = t if t % 2 else t + 1
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(1, 1, k, padding=(k - 1) // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        y = self.avg_pool(x).squeeze(-1).transpose(-1, -2)
        y = self.sigmoid(self.conv(y)).transpose(-1, -2).unsqueeze(-1)
        return x * y.expand_as(x)


class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super().__init__()
        mid = max(1, in_channels // reduction)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(
            nn.Conv2d(in_channels, mid, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid, in_channels, 1, bias=False),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        return self.sigmoid(self.mlp(self.avg_pool(x)) + self.mlp(self.max_pool(x)))


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg = torch.mean(x, 1, keepdim=True)
        mx, _ = torch.max(x, 1, keepdim=True)
        return self.sigmoid(self.conv(torch.cat([avg, mx], 1)))


class ECA_Only(nn.Module):
    "Ablation B -- channel only."

    def __init__(self, in_channels):
        super().__init__()
        self.eca = ECA(in_channels)
        self.scale = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        return x + torch.tanh(self.scale) * (self.eca(x) - x)


class CBAM_Only(nn.Module):
    "Ablation C -- spatial+channel, no ECA pre-stage."

    def __init__(self, in_channels, reduction=16, kernel_size=7):
        super().__init__()
        self.ca = ChannelAttention(in_channels, reduction)
        self.sa = SpatialAttention(kernel_size)
        self.scale = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        att = x * self.ca(x)
        att = att * self.sa(att)
        return x + torch.tanh(self.scale) * (att - x)


class ECA_CBAM(nn.Module):
    "Main model -- ECA then CBAM, residual scale init at 0."

    def __init__(self, in_channels, cbam_reduction=16, cbam_kernel=7):
        super().__init__()
        self.eca = ECA(in_channels)
        self.ca = ChannelAttention(in_channels, cbam_reduction)
        self.sa = SpatialAttention(cbam_kernel)
        self.scale = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        att = self.eca(x)
        att = att * self.ca(att)
        att = att * self.sa(att)
        return x + torch.tanh(self.scale) * (att - x)


dummy = torch.randn(2, 256, 20, 20)
for cls in [ECA_Only, CBAM_Only, ECA_CBAM]:
    m = cls(256)
    out = m(dummy)
    n_p = sum(p.numel() for p in m.parameters())
    diff = (out - dummy).abs().max().item()
    print(f"{cls.__name__:<12}: params={n_p:,}  init_diff={diff:.6f} (should be 0)")


ECA_Only    : params=6  init_diff=0.000000 (should be 0)
CBAM_Only   : params=8,291  init_diff=0.000000 (should be 0)
ECA_CBAM    : params=8,296  init_diff=0.000000 (should be 0)


 Hook Injector

In [8]:
# Defines get_neck_modules and AttentionHookInjector, which locate the neck layers and attach the attention modules as forward hooks.
def get_neck_modules(yolo_detection_model):
    if hasattr(yolo_detection_model, "model") and isinstance(
        yolo_detection_model.model, nn.Sequential
    ):
        layer_seq = yolo_detection_model.model
    else:
        children = list(yolo_detection_model.children())
        layer_seq = next((c for c in children if isinstance(c, nn.Sequential)), None)
        if layer_seq is None:
            raise RuntimeError("Cannot locate Sequential inside DetectionModel")
    NECK_NAMES = {"C3k2", "C2f", "C2fAttn", "RepC3", "C3"}
    candidates = [
        (i, l, type(l).__name__)
        for i, l in enumerate(list(layer_seq))
        if type(l).__name__ in NECK_NAMES
    ]
    print(f"  Total layers: {len(list(layer_seq))}  neck candidates: {len(candidates)}")
    if not candidates:
        raise RuntimeError(
            f"No neck layers. Classes: {set(type(l).__name__ for l in list(layer_seq))}"
        )
    return [
        l for (_, l, _) in (candidates[-3:] if len(candidates) >= 3 else candidates)
    ]


class AttentionHookInjector:
    def __init__(
        self, yolo_detection_model, attention_cls, cbam_reduction=16, cbam_kernel=7
    ):
        self._hooks = []
        self.attention_modules = nn.ModuleList()
        self._device = next(yolo_detection_model.parameters()).device

        neck_layers = get_neck_modules(yolo_detection_model)
        real_channels = [None] * len(neck_layers)
        probe_hooks = []

        def make_probe(i):
            def hook(module, inp, out):
                t = out[0] if isinstance(out, (list, tuple)) else out
                real_channels[i] = t.shape[1]

            return hook

        for i, layer in enumerate(neck_layers):
            probe_hooks.append(layer.register_forward_hook(make_probe(i)))
        try:
            dummy = torch.zeros(1, 3, 640, 640, device=self._device)
            with torch.no_grad():
                yolo_detection_model(dummy)
        except Exception as e:
            print(f"  Probe note: {e}")
        finally:
            for h in probe_hooks:
                h.remove()

        for i, (layer, c) in enumerate(zip(neck_layers, real_channels)):
            if c is None:
                print(f"  Layer {i}: probe failed")
                continue
            if attention_cls == ECA_Only:
                att = ECA_Only(c).to(self._device)
            elif attention_cls == CBAM_Only:
                att = CBAM_Only(c, cbam_reduction, cbam_kernel).to(self._device)
            else:
                att = ECA_CBAM(c, cbam_reduction, cbam_kernel).to(self._device)
            self.attention_modules.append(att)
            n_p = sum(p.numel() for p in att.parameters())
            print(f"  Neck layer {i}: ch={c}  {attention_cls.__name__}  params={n_p:,}")
        self._neck_layers = neck_layers

    def attach(self):
        self._hooks = []
        for layer, att in zip(self._neck_layers, self.attention_modules):

            def make_hook(a):
                def hook(module, inp, out):
                    if isinstance(out, (list, tuple)):
                        return type(out)([a(out[0])] + list(out[1:]))
                    return a(out)

                return hook

            self._hooks.append(layer.register_forward_hook(make_hook(att)))
        print(f"  {len(self._hooks)} hooks attached")

    def detach(self):
        for h in self._hooks:
            h.remove()
        self._hooks = []
        print("  Hooks removed")

    def n_params(self):
        return sum(p.numel() for p in self.attention_modules.parameters())

    def state_dict(self):
        return self.attention_modules.state_dict()


print("AttentionHookInjector defined")


AttentionHookInjector defined
